# Extracting raw MHR parameters under the hood

By default, the application only returns a series of `.ply` files containing the mesh of a person at each frame.  
By inspecting the GitHub code:

- Inside the `SAM3DBodyEstimator` class, the `process_frames` method returns `all_out_batch`,  
  which is structured as `List[List[Dict]]`, specifically,

<div align="center">
  <code>all_out_batch[frame_index][person_index] → dict</code>
</div>

- Here, `dict` looks like:

```python
{
    "bbox": batch_dict["bbox"][b_idx, idx].cpu().numpy(),
    "focal_length": out["focal_length"][b_idx * num_objects + idx],
    "pred_keypoints_3d": out["pred_keypoints_3d"][b_idx * num_objects + idx],
    "pred_keypoints_2d": out["pred_keypoints_2d"][b_idx * num_objects + idx],
    "pred_vertices": out["pred_vertices"][b_idx * num_objects + idx],
    "pred_cam_t": out["pred_cam_t"][b_idx * num_objects + idx],
    "pred_pose_raw": out["pred_pose_raw"][b_idx * num_objects + idx],
    "global_rot": out["global_rot"][b_idx * num_objects + idx],
    "body_pose_params": out["body_pose"][b_idx * num_objects + idx],
    "hand_pose_params": out["hand"][b_idx * num_objects + idx],
    "scale_params": out["scale"][b_idx * num_objects + idx],
    "shape_params": out["shape"][b_idx * num_objects + idx],
    "expr_params": out["face"][b_idx * num_objects + idx],
    "mask": masks[b_idx] if masks is not None else None,
    "pred_joint_coords": out["pred_joint_coords"][b_idx * num_objects + idx],
    "pred_global_rots": out["joint_global_rots"][b_idx * num_objects + idx],
    "mhr_model_params": out["mhr_model_params"][b_idx * num_objects + idx],
}
```

- In `process_image_wth_mask`, after a bunch of stuff, `final_outputs` is returned (as the first tupple element) which has the same structure as `all_out_batch` except the frames are temporally ordered (which all_out_batch doesn't require) and occlusion handing is only applied when needed (much faster!)

```python
final_outputs = [
    [ # Frame 0
        {...}, # Person A
        {...}, # Person B
        ...
    ],
    [ # Frame 1
        {...}, # Person A
        {...}, # Person B
        ...
    ],
    ...
]
```

# Extracting raw MHR parameters in the application

In `offline_app.py`, the following function call was made:

<div align="center">
  <code>mask_outputs, id_batch, empty_frame_list = process_image_with_mask(...)</code>
</div>

So the MHR parameters are contained in `mask_outputs`, which is then passed into `save_mesh_results` to save a series of .ply file.

On top of the raw HMR parameters, I should also store camera position and focal length because they are used when reconstructing the mesh. Raw HMR parameters may or may not be enough.